# 00 setup

Run once per Colab session. Runtime -> Change runtime type -> T4 GPU.

Everything on Drive lives under `MyDrive/Colab Notebooks/tfm/` (`ROOT` below). This notebook sits in `ROOT/notebooks/` and is opened from Drive with Colab. Code is pulled from GitHub with a single-repo fine-grained token (Contents: read and write, 30-day expiry) stored in Colab Secrets as `GH_TOKEN`. The push cell copies small results and the notebooks (outputs cleared) back into the repo. Checkpoints and data stay on Drive.

In [1]:
import os, sys, glob, shutil
from google.colab import drive, userdata
drive.mount("/content/drive")
ROOT = "/content/drive/MyDrive/Colab Notebooks/tfm"
tok = userdata.get("GH_TOKEN")
if not os.path.exists("/content/transaction-fm"):
    !git clone -q https://{tok}@github.com/lb603487485/transaction-fm.git /content/transaction-fm
else:
    !cd /content/transaction-fm && git pull -q
%pip install -q datasets xgboost
%cd /content/transaction-fm
sys.path.insert(0, "/content/transaction-fm/src")
os.makedirs(f"{ROOT}/notebooks", exist_ok=True)
for f in glob.glob("notebooks/*.ipynb"):          # new notebooks from the repo appear in Drive
    dst = f"{ROOT}/notebooks/{os.path.basename(f)}"
    if not os.path.exists(dst):
        shutil.copy(f, dst); print("new notebook in Drive:", os.path.basename(f))
import tfm
print("tfm ok")

Mounted at /content/drive
/content/transaction-fm
new notebook in Drive: 01_data.ipynb
tfm ok


## Check the GPU and the Drive layout

In [ ]:
import os, torch
print("gpu:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
for d in ["notebooks", "data/raw", "data/processed", "pretrain", "finetune", "xgb", "results"]:
    p = f"{ROOT}/{d}"
    os.makedirs(p, exist_ok=True)
    print(f"{d:16s}", sorted(os.listdir(p))[:10])

`data/raw` should list the two Sparkov csv files (`fraudTrain.csv`, `fraudTest.csv`) before step 1.

## Push results and notebooks back to GitHub

Run after a step has written json or png into `ROOT/results/`. Copies the small result files and every notebook in `ROOT/notebooks/` (outputs cleared) into the repo, then pushes. Checkpoints never leave Drive.

In [ ]:
import shutil, glob, os, json
os.makedirs("results/metrics", exist_ok=True); os.makedirs("results/figures", exist_ok=True)
for f in glob.glob(f"{ROOT}/results/*.json"):
    shutil.copy(f, "results/metrics/")
for f in glob.glob(f"{ROOT}/results/*.png"):
    shutil.copy(f, "results/figures/")
for f in glob.glob(f"{ROOT}/notebooks/*.ipynb"):
    nb = json.load(open(f))
    for c in nb["cells"]:
        if c["cell_type"] == "code":
            c["outputs"] = []; c["execution_count"] = None
    json.dump(nb, open(os.path.join("notebooks", os.path.basename(f)), "w"), indent=1, ensure_ascii=False)
!git config user.email "linboyy@hotmail.com" && git config user.name "Bo Lin"
!git add results notebooks && git commit -q -m "results and notebooks from Colab" && git push -q
!git log --oneline | head -1

## End of session

In [ ]:
drive.flush_and_unmount()